# Load exact training labels (y) for a split

This notebook shows a small helper `load_split_labels` that reconstructs the exact sequence of labels used by the trainer for a given split JSON (train/val/test). It mirrors the trainer behavior: it prefers `y_path` when present and otherwise reads the `y` array from the referenced `.npz` files.

In [1]:
import json
from pathlib import Path
import numpy as np
from typing import List

def load_split_labels(split_json_path: str | Path, split_name: str = "train") -> np.ndarray:
    """Return a 1-D numpy array of labels in the same order the training dataset would consume them.

    The function reads the `*_items` list from the split JSON. For each item it uses `y_path` if provided, otherwise it opens the referenced `.npz` and reads the `y` array. For items with `patch_index`==None it will extend with all labels from that file (this is how augmented `.npz` files are represented).
    """
    split = json.loads(Path(split_json_path).read_text(encoding="utf-8"))
    items = split.get(f"{split_name}_items", [])
    if not items:
        raise ValueError(f"No '{split_name}_items' in {split_json_path}")

    y_cache: dict[str, np.ndarray] = {}
    out_labels: List[int] = []

    for item in items:
        npz_path = Path(item["npz_path"])
        # use y_path when present, otherwise key uses npz path so shared files are cached once
        key = str(item.get("y_path", npz_path))
        if key not in y_cache:
            if item.get("y_path"):
                y_cache[key] = np.load(item["y_path"], allow_pickle=True).astype(int)
            else:
                with np.load(npz_path, allow_pickle=True) as d:
                    if "y" in d:
                        y_cache[key] = d["y"].astype(int)
                    elif "labels" in d:
                        y_cache[key] = d["labels"].astype(int)
                    else:
                        raise KeyError(f"No 'y' array found in {npz_path}")

        y_arr = np.asarray(y_cache[key]).reshape(-1)
        patch_index = item.get("patch_index", None)
        if patch_index is None or (isinstance(patch_index, str) and patch_index.lower() == "none"):
            out_labels.extend(int(x) for x in y_arr)
        else:
            idx = int(patch_index)
            out_labels.append(int(y_arr[idx]))

    return np.asarray(out_labels, dtype=int)

In [2]:
# Usage example - update the path to a real split.json in your cache
from pathlib import Path

# Example path (adjust to your environment):
example_split = Path("D:/data/radius_search_plus/_cache/radius_1km/norwegian_only/fold_0_test_0/split.json")

if example_split.exists():
    y_train = load_split_labels(example_split, "train")
    y_val = load_split_labels(example_split, "val")
    y_test = load_split_labels(example_split, "test")

    print("train:", y_train.shape, "positives:", int(y_train.sum()), "rate:", float(y_train.mean()))
    print("val:", y_val.shape, "positives:", int(y_val.sum()), "rate:", float(y_val.mean()))
    print("test:", y_test.shape, "positives:", int(y_test.sum()), "rate:", float(y_test.mean()))
else:
    print("Example split.json not found - update 'example_split' path to a real split.json in your cache.")

FileNotFoundError: [Errno 2] No such file or directory: '/mnt/e/data/processed/norwegian/norwegian/260326_part1/loc_1/loc_1_260326_part1_fold2_y_radius1.0km.npy'

Notes:
- This reproduces exactly the label ordering used by the training dataset builder (it reads per-item `y_path` when provided, otherwise `y` inside `.npz`).
- For augmented `.npz` files where `patch_index` is `null` the code includes all labels from that file (same as training).
- If your `.npz` uses a different label key, update the key checks in the helper accordingly.